In [ ]:
import matplotlib.pyplot as plt
from pandas import DataFrame
from common.utils import load_dataset, optimize_memory, get_params, DatasetType
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import seaborn as sns
from plotnine import ggplot, aes, geom_histogram, scale_x_log10, scale_y_sqrt
import pandas as pd
import matplotlib.ticker as mtick
import calendar

In [ ]:
df: DataFrame = load_dataset("nyc-taxi-trip-duration", DatasetType.TRAIN, index=True)
df.head()

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.describe()

#### Missing values

In [ ]:
missing_counts = df.isna().sum()
missing_counts.plot(kind='bar')
plt.ylabel('Number of Missing Values')
plt.title('Missing Values per Column')
plt.show()

In [ ]:
df, na_list = optimize_memory(df, deep=True)
df.dtypes

In [ ]:
df.head()

In [ ]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['dropoff_datetime'] = pd.to_datetime(df['dropoff_datetime'])
df.head()

In [ ]:
df.dtypes

##### Validate the trip duration with the pickup and drop off time

In [ ]:
df['check'] = ((df['dropoff_datetime'] - df['pickup_datetime']).dt.total_seconds()
               + df['trip_duration']).abs() > 0

result = df[['check', 'pickup_datetime', 'dropoff_datetime', 'trip_duration']] \
    .groupby('check') \
    .size() \
    .reset_index(name='n')

print(result)

##### Trip duration plot

In [ ]:
(
    ggplot(df, aes(x='trip_duration'))
    + geom_histogram(fill='red', bins=150)
    + scale_x_log10()
    + scale_y_sqrt()
)

In [ ]:
df_sorted = df.sort_values('trip_duration', ascending=False)

cols = ['trip_duration', 'pickup_datetime', 'dropoff_datetime'] + \
       [col for col in df.columns if col not in ['trip_duration', 'pickup_datetime', 'dropoff_datetime']]
df_sorted = df_sorted[cols]
df_sorted.head(10)

##### Trip duration by month

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 6), sharex=False)

axes[0].hist(df['pickup_datetime'], bins=120, color='red')
axes[0].set_xlabel('Pickup dates')
axes[0].set_ylabel('Count')

axes[1].hist(df['dropoff_datetime'], bins=120, color='blue')
axes[1].set_xlabel('Dropoff dates')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
mask = (df['pickup_datetime'] > pd.to_datetime("2016-01-20")) & \
       (df['pickup_datetime'] < pd.to_datetime("2016-02-10"))

filtered_df = df.loc[mask]

plt.figure(figsize=(10, 5))
plt.hist(filtered_df['pickup_datetime'], bins=120, color='red')
plt.xlabel('Pickup Datetime')
plt.ylabel('Count')
plt.title('Pickup Time Distribution')
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 12))
axes = axes.flatten()

passenger_counts = df.groupby('passenger_count').size().reset_index(name='n')


passenger_counts['n_sqrt'] = np.sqrt(passenger_counts['n'])

sns.barplot(x='passenger_count', y='n_sqrt', data=passenger_counts,
            hue='passenger_count', dodge=False, ax=axes[0])


sns.barplot(x='passenger_count', y='n', data=passenger_counts,
            hue='passenger_count', dodge=False, ax=axes[0])

#axes[0].set_yscale('sqrt')
axes[0].legend().remove()

sns.countplot(x='vendor_id', hue='vendor_id', data=df, ax=axes[1])
axes[1].legend().remove()

sns.countplot(x='store_and_fwd_flag', data=df, ax=axes[2])
axes[2].set_yscale('log')
axes[2].legend().remove()

df['wday'] = df['pickup_datetime'].dt.day_name()
wday_counts = df.groupby(['wday', 'vendor_id']).size().reset_index(name='n')
sns.scatterplot(x='wday', y='n', hue='vendor_id', data=wday_counts,
                s=100, ax=axes[3])
axes[3].set_xlabel('Day of the week')
axes[3].set_ylabel('Total number of pickups')
axes[3].legend().remove()

df['hpick'] = df['pickup_datetime'].dt.hour
hpick_counts = df.groupby(['hpick', 'vendor_id']).size().reset_index(name='n')
sns.scatterplot(x='hpick', y='n', hue='vendor_id', data=hpick_counts,
                s=100, ax=axes[4])
axes[4].set_xlabel('Hour of the day')
axes[4].set_ylabel('Total number of pickups')
axes[4].legend().remove()

axes[5].axis('off')

plt.tight_layout()
plt.show()

##### Rides by passenger count

In [ ]:
counts = df['passenger_count'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(counts.index.astype(str), counts.values)

ax.set_xlabel('Passenger Count')
ax.set_ylabel('Count')
ax.set_title('Rides per passengers count')

ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))

for bar in bars:
    h = bar.get_height()
    ax.annotate(f'{int(h):,}',
                xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
counts = df['store_and_fwd_flag'].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(counts.index.astype(str), counts.values, color='skyblue')

ax.set_xlabel('store_and_fwd_flag')
ax.set_ylabel('Count')
ax.set_title('Store and Forward Flag Distribution')

ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))

for bar in bars:
    h = bar.get_height()
    ax.annotate(f'{int(h):,}',
                xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

df['hpick'] = df['pickup_datetime'].dt.hour

months = list(calendar.month_abbr)[1:]
df['Month'] = pd.Categorical(
    df['pickup_datetime'].dt.strftime('%b'),
    categories=months, ordered=True
)

group_month = (
    df.groupby(['hpick', 'Month'])
      .size()
      .reset_index(name='n')
)

weekdays = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
df['wday'] = pd.Categorical(
    df['pickup_datetime'].dt.strftime('%a'),
    categories=weekdays, ordered=True
)

group_wday = (
    df.groupby(['hpick', 'wday'])
      .size()
      .reset_index(name='n')
)

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True, gridspec_kw={'height_ratios':[1,1]})

sns.lineplot(data=group_month, x='hpick', y='n', hue='Month', ax=axes[0], linewidth=1.5)
axes[0].set_title('Pickups by Hour — Colored by Month')
axes[0].set_xlabel('')
axes[0].set_ylabel('count')
axes[0].legend(title='Month', bbox_to_anchor=(1.02, 1), loc='upper left')

sns.lineplot(data=group_wday, x='hpick', y='n', hue='wday', ax=axes[1], linewidth=1.5)
axes[1].set_title('Pickups by Hour — Colored by Weekday')
axes[1].set_xlabel('Hour of the day')
axes[1].set_ylabel('count')
axes[1].legend(title='Weekday', bbox_to_anchor=(1.02, 1), loc='upper left')

for ax in axes:
    ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))
    ax.set_xticks(range(0, 24))
    ax.set_xlim(-0.5, 23.5)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

pickup_long_filter = df[(df['pickup_longitude'] > -74.05) & (df['pickup_longitude'] < -73.7)]
sns.histplot(pickup_long_filter['pickup_longitude'], bins=40, color='red', ax=axes[0, 0])
axes[0, 0].set_title('Pickup Longitude')

dropoff_long_filter = df[(df['dropoff_longitude'] > -74.05) & (df['dropoff_longitude'] < -73.7)]
sns.histplot(dropoff_long_filter['dropoff_longitude'], bins=40, color='blue', ax=axes[0, 1])
axes[0, 1].set_title('Dropoff Longitude')

pickup_lat_filter = df[(df['pickup_latitude'] > 40.6) & (df['pickup_latitude'] < 40.9)]
sns.histplot(pickup_lat_filter['pickup_latitude'], bins=40, color='red', ax=axes[1, 0])
axes[1, 0].set_title('Pickup Latitude')

dropoff_lat_filter = df[(df['dropoff_latitude'] > 40.6) & (df['dropoff_latitude'] < 40.9)]
sns.histplot(dropoff_lat_filter['dropoff_latitude'], bins=40, color='blue', ax=axes[1, 1])
axes[1, 1].set_title('Dropoff Latitude')

for ax in axes.flatten():
    ax.set_xlabel('')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()